# 03 | GPU Data Curation & SFT Dataset Preparation (Live Demo)

**Objective:** Stress-test and clean raw transcripts using **NVIDIA NeMo Curator**, track comprehensive pipeline & data metrics via **Weights & Biases (W&B)**, and prepare final Supervised Fine-Tuning (SFT) splits.

**End-to-End Pipeline & Lineage:**
```text
raw-transcripts:latest ──► [1. Noise Injection] ──► [2. NeMo Curator (GPU) + W&B] ──► [3. SFT Splits Artifact]
 (Consumed via use_artifact)    (PII, Dupes, Bugs)       (Masking, Filter, Dedup)         (curated-sft-dataset:latest)
```

## 1. W&B Lineage Connection (`use_artifact`), Noise & PII Injection
Dynamically consumes the versioned `raw-transcripts:latest` artifact from the previous Data Generation stage, establishes strict pipeline lineage, and simulates real-world data quality issues.

In [1]:
import json
import random
from pathlib import Path
import wandb

# Initialize Weights & Biases Run with job_type for End-to-End Lineage
run = wandb.init(
    project="nemo-fraud-detection",
    job_type="data-curation",
    name="data-curation-and-sft-splits-advanced",
    tags=["stage-2-curation", "nemo-curator", "lineage"],
    config={
        "pipeline": "nemo-curator",
        "min_length_filter": 50,
        "train_split_ratio": 0.70,
        "val_split_ratio": 0.15,
        "test_split_ratio": 0.15,
        "input_artifact": "raw-transcripts:latest"
    }
)

# Consume upstream artifact to establish explicit End-to-End Lineage in W&B
print("📥 Downloading upstream artifact 'raw-transcripts:latest'...")
artifact_dir = run.use_artifact("raw-transcripts:latest").download()
RAW_PATH = Path(artifact_dir) / "transcripts.jsonl"
BENCHMARK_PATH = Path(artifact_dir) / "benchmark.jsonl"

BASE_DIR = Path("/data/nemo-fraud-detection-notebooks/notebooks")
NOISY_PATH = BASE_DIR / "02_Data_Curation/data/transcripts_noisy.jsonl"
NOISY_PATH.parent.mkdir(parents=True, exist_ok=True)

PII_SAMPLES = [
    "My credit card number is 4532-8921-1029-4411 with CVV 892.",
    "Reach me at john.doe@example.com or +49 171 1234567.",
    "My IBAN is DE89 3704 0044 0532 0130 00, account holder Thomas Miller."
]
GARBAGE_SNIPPETS = ["???", "[ERROR_VOICEMAIL_CORRUPTED]", "CLICK... BEEP..."]

def inject_noise():
    records = []
    with open(RAW_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    continue

    noisy_dataset = []
    stats = {"duplicate": 0, "pii": 0, "garbage": 0, "encoding": 0}
    random.seed(42)

    for idx, item in enumerate(records):
        cid = item.get("id", f"CALL_{idx:04d}")
        text = item.get("text", "")
        dice = random.random()

        if dice < 0.15:
            noisy_dataset.extend([{"call_id": cid, "text": text}, {"call_id": f"{cid}_DUP", "text": text}])
            stats["duplicate"] += 1
        elif dice < 0.35:
            noisy_dataset.append({"call_id": cid, "text": f"{text}\nCustomer: {random.choice(PII_SAMPLES)}"})
            stats["pii"] += 1
        elif dice < 0.45:
            noisy_dataset.append({"call_id": cid, "text": random.choice(GARBAGE_SNIPPETS)})
            stats["garbage"] += 1
        elif dice < 0.55:
            noisy_dataset.append({"call_id": cid, "text": text.replace("a", "Ã¤").replace("o", "Ã¶")})
            stats["encoding"] += 1
        else:
            noisy_dataset.append({"call_id": cid, "text": text})

    with open(NOISY_PATH, "w", encoding="utf-8") as f:
        for rec in noisy_dataset:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    # Log injection noise stats to W&B
    run.log({
        "curation/injected_duplicates": stats["duplicate"],
        "curation/injected_pii_leaks": stats["pii"],
        "curation/injected_garbage": stats["garbage"],
        "curation/injected_encoding_errors": stats["encoding"]
    })

    print(f"🧬 Injecting realistic noise into {len(records)} clean transcripts from upstream artifact...")
    print(f"📊 Injection Summary: {stats['duplicate']} Duplicates | {stats['pii']} PII Leaks | {stats['garbage']} Garbage | {stats['encoding']} Encoding Error")
    print(f"✅ Created {len(noisy_dataset)} noisy test records -> {NOISY_PATH.name}")

inject_noise()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /root/.netrc.
wandb: Currently logged in as: lohmann-daniel (lohmann-daniel-myself) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


📥 Downloading upstream artifact 'raw-transcripts:latest'...


wandb:   2 of 2 files downloaded.  


🧬 Injecting realistic noise into 93 clean transcripts from upstream artifact...
📊 Injection Summary: 14 Duplicates | 17 PII Leaks | 13 Garbage | 7 Encoding Error
✅ Created 107 noisy test records -> transcripts_noisy.jsonl


## 2. GPU-Accelerated Curation with NVIDIA NeMo Curator & W&B Metrics
Executes a 4-stage curation pipeline on GPU and logs detailed cleaning and removal metrics directly via `run.log`:

In [2]:
import os
import re
import shutil
from typing import Any
from nemo_curator import Modify, ScoreFilter, Sequential, AddId
from nemo_curator.datasets import DocumentDataset
from nemo_curator.filters import DocumentFilter
from nemo_curator.modules import ExactDuplicates
from nemo_curator.utils.distributed_utils import get_client
from nemo_curator.modifiers import DocumentModifier, UnicodeReformatter

os.environ["KVIKIO_COMPAT_MODE"] = "ON"
os.environ["CUDF_CUFILE_ENABLED"] = "0"
os.environ["RAPIDS_NO_CUFILE"] = "1"
os.environ["DASK_DATAFRAME__QUERY_PLANNING"] = "False"

CURATED_PATH = BASE_DIR / "02_Data_Curation/data/transcripts_curated.jsonl"
TEMP_DIR = BASE_DIR / "02_Data_Curation/data/_temp_export"
LOG_DIR = BASE_DIR / "02_Data_Curation/data/dedup_logs"
CACHE_DIR = BASE_DIR / "02_Data_Curation/data/dedup_cache"

for d in [CURATED_PATH.parent, LOG_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

class FraudPiiModifier(DocumentModifier):
    def modify_document(self, doc: str) -> str:
        if not isinstance(doc, str): return doc
        doc = re.sub(r'DE\d{2}\s?(\d{4}\s?){4}\d{2}', '[IBAN_MASKED]', doc)
        doc = re.sub(r'\b(?:\d[ -]*?){13,16}\b', '[CREDIT_CARD_MASKED]', doc)
        doc = re.sub(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', '[EMAIL_MASKED]', doc)
        doc = re.sub(r'\b(?:\+49|0)\s*1[567]\d[\s\-]?\d{3,8}\b', '[PHONE_MASKED]', doc)
        return doc

class MinLengthFilter(DocumentFilter):
    def __init__(self, min_length: int = 50):
        super().__init__()
        self.min_length = min_length
    def score_document(self, doc: Any) -> int:
        text = doc.get("text", "") if isinstance(doc, dict) else str(doc)
        return len(text.strip())
    def keep_document(self, score: int) -> bool:
        return score >= self.min_length

def run_curator():
    print("🚀 Starting NVIDIA NeMo Curator GPU Pipeline...")
    client = get_client(cluster_type="gpu", set_torch_to_use_rmm=False)
    dataset = DocumentDataset.read_json(str(NOISY_PATH), add_filename=True, backend="pandas")
    initial_count = len(dataset.df)

    print("🛡️ Running Unicode Reformatter + PII Masking + Length Filter + MD5 Deduplication...")
    dataset = AddId(id_field="id", id_prefix="FRAUD_data", start_index=0)(dataset)
    dataset = Sequential([Modify(UnicodeReformatter()), Modify(FraudPiiModifier())])(dataset).persist()
    dataset = ScoreFilter(MinLengthFilter(min_length=50), score_type=int)(dataset)

    exact_dup = ExactDuplicates(logger=str(LOG_DIR), id_field="id", text_field="text", hash_method="md5", cache_dir=str(CACHE_DIR))
    dups = exact_dup(dataset=dataset)
    to_remove = dups.df.map_partitions(lambda x: x[x._hashes.duplicated(keep="first")])["id"].compute()
    dataset = DocumentDataset(dataset.df[~dataset.df["id"].isin(to_remove)])

    if TEMP_DIR.exists(): shutil.rmtree(TEMP_DIR)
    if CURATED_PATH.exists(): CURATED_PATH.unlink()
    dataset.to_json(str(TEMP_DIR), write_to_filename=False)

    with open(CURATED_PATH, "w", encoding="utf-8") as outfile:
        for part in sorted(TEMP_DIR.iterdir()):
            if part.is_file():
                with open(part, "r", encoding="utf-8") as infile:
                    shutil.copyfileobj(infile, outfile)
    shutil.rmtree(TEMP_DIR)

    with open(CURATED_PATH, "r", encoding="utf-8") as f:
        final_count = sum(1 for line in f if line.strip())

    removed_count = initial_count - final_count
    removal_rate = (removed_count / initial_count) * 100 if initial_count > 0 else 0

    # Log comprehensive curation metrics to W&B
    run.log({
        "curation/input_records": initial_count,
        "curation/removed_records": removed_count,
        "curation/clean_output_records": final_count,
        "curation/removal_rate_pct": removal_rate,
    })

    print("\n" + "="*60)
    print("📊 NEMO CURATOR PIPELINE SUMMARY")
    print("="*60)
    print(f"  • Input Records (Noisy):       {initial_count}")
    print(f"  • Removed (Garbage/Dupes):     {removed_count}")
    print(f"  • Clean & Anonymized Output:   {final_count}")
    print(f"  • Removal Rate:                {removal_rate:.1f}%")
    print("="*60)

run_curator()

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🚀 Starting NVIDIA NeMo Curator GPU Pipeline...
Reading 1 files
🛡️ Running Unicode Reformatter + PII Masking + Length Filter + MD5 Deduplication...


/opt/NeMo-Curator/nemo_curator/modules/exact_dedup.py:158: UserWarning: Output path f/data/nemo-fraud-detection-notebooks/notebooks/02_Data_Curation/data/dedup_cache/_exact_duplicates.parquet already exists and will be overwritten
  warnings.warn(


Writing to disk complete for 1 partitions

📊 NEMO CURATOR PIPELINE SUMMARY
  • Input Records (Noisy):       107
  • Removed (Garbage/Dupes):     27
  • Clean & Anonymized Output:   80
  • Removal Rate:                25.2%


## 3. Before vs. After Inspection (PII Redaction Proof)
Demonstrating how sensitive PII injected in Step 1 is automatically redacted in the curated output, rendered inside an einklappbare (collapsible) HTML UI element.

In [3]:
import html
from IPython.display import display, HTML

with open(NOISY_PATH, "r", encoding="utf-8") as f:
    noisy_recs = {json.loads(l)["call_id"]: json.loads(l)["text"] for l in f if l.strip()}
with open(CURATED_PATH, "r", encoding="utf-8") as f:
    curated_recs = {json.loads(l)["call_id"]: json.loads(l)["text"] for l in f if l.strip()}

comparison_content = ""
for cid, clean_txt in curated_recs.items():
    if "_MASKED]" in clean_txt:
        comparison_content += "🔴 BEFORE CURATION (Raw Noisy Transcript):\n"
        comparison_content += noisy_recs[cid] + "\n\n"
        comparison_content += "🟢 AFTER NEMO CURATOR (Anonymized & Cleaned):\n"
        comparison_content += clean_txt + "\n"
        break

escaped_comp = html.escape(comparison_content)
html_widget = f"""
<details style="border: 1px solid #d0d7de; border-radius: 6px; padding: 10px 14px; background-color: #f6f8fa; margin-top: 10px;" open>
    <summary style="cursor: pointer; font-weight: bold; color: #0969da; font-size: 14px;">
        ▶ Sample Output (Before vs. After PII Redaction)
    </summary>
    <pre style="margin-top: 12px; white-space: pre-wrap; font-size: 13px; font-family: monospace; background: #ffffff; padding: 10px; border: 1px solid #e1e4e8; border-radius: 4px;">{escaped_comp}</pre>
</details>
"""
display(HTML(html_widget))

## 4. Supervised Fine-Tuning (SFT) Dataset Export & W&B Artifacts Lineage
Matches curated transcripts with ground-truth labels from the benchmark file, generates stratified splits, logs split sizes to W&B, publishes them as the versioned `curated-sft-dataset` artifact, and concludes the run.

In [4]:
SFT_DIR = BASE_DIR / "02_Data_Curation/data/sft"
SFT_DIR.mkdir(parents=True, exist_ok=True)

with open(BENCHMARK_PATH, "r", encoding="utf-8") as f:
    gt_map = {str(json.loads(l)["id"]): json.loads(l)["label"] for l in f if l.strip()}

with open(CURATED_PATH, "r", encoding="utf-8") as f:
    curated = [json.loads(l) for l in f if l.strip()]

sft_samples = [{"input": d["text"], "output": gt_map[str(d["call_id"])]} for d in curated if str(d.get("call_id")) in gt_map]

random.seed(42)
random.shuffle(sft_samples)

n = len(sft_samples)
train_end = int(n * 0.70)
val_end = train_end + int(n * 0.15)

splits = {"train.jsonl": sft_samples[:train_end], "validation.jsonl": sft_samples[train_end:val_end], "test.jsonl": sft_samples[val_end:]}

for filename, data in splits.items():
    with open(SFT_DIR / filename, "w", encoding="utf-8") as f:
        for row in data:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

# Log SFT split metrics to W&B
run.log({
    "sft/train_samples": len(splits["train.jsonl"]),
    "sft/validation_samples": len(splits["validation.jsonl"]),
    "sft/test_samples": len(splits["test.jsonl"])
})

print(f"✅ Exported SFT Splits to {SFT_DIR}:")
print(f"   • train.jsonl      : {len(splits['train.jsonl'])} samples (70%)")
print(f"   • validation.jsonl : {len(splits['validation.jsonl'])} samples (15%)")
print(f"   • test.jsonl       : {len(splits['test.jsonl'])} samples (15%)")

# Upload SFT splits as a W&B Artifact with End-to-End Lineage metadata and finish run
print("📦 Uploading curated SFT dataset splits as artifact 'curated-sft-dataset' to Weights & Biases...")
artifact = wandb.Artifact(
    name="curated-sft-dataset",
    type="dataset",
    description="Curated, PII-masked and stratified 70/15/15 SFT splits for banking fraud detection",
    metadata={
        "stage": "02_data_curation",
        "train_samples": len(splits["train.jsonl"]),
        "validation_samples": len(splits["validation.jsonl"]),
        "test_samples": len(splits["test.jsonl"])
    }
)
artifact.add_dir(str(SFT_DIR))

logged_artifact = run.log_artifact(artifact, aliases=["latest", "curated-v1"])
logged_artifact.wait()

run.finish()
print("✅ W&B data curation lineage registered successfully!")

wandb: Adding directory to artifact (/data/nemo-fraud-detection-notebooks/notebooks/02_Data_Curation/data/sft)... Done. 0.0s


✅ Exported SFT Splits to /data/nemo-fraud-detection-notebooks/notebooks/02_Data_Curation/data/sft:
   • train.jsonl      : 56 samples (70%)
   • validation.jsonl : 12 samples (15%)
   • test.jsonl       : 12 samples (15%)
📦 Uploading curated SFT dataset splits as artifact 'curated-sft-dataset' to Weights & Biases...


curation/clean_output_records,▁
curation/injected_duplicates,▁
curation/injected_encoding_errors,▁
curation/injected_garbage,▁
curation/injected_pii_leaks,▁
curation/input_records,▁
curation/removal_rate_pct,▁
curation/removed_records,▁
sft/test_samples,▁
sft/train_samples,▁
+1,...


✅ W&B data curation lineage registered successfully!


### Key Takeaways & Next Steps
* **End-to-End Lineage:** Seamlessly consumed `raw-transcripts:latest` and published `curated-sft-dataset:latest`.
* **Production-Grade Data Hygiene:** PII redaction, encoding fixes, and deduplication happen automatically on GPU.